In [1]:
NUM_EPOCH = 30
ETA = 1.0
MINI_BATCH_SIZE = 10

In [2]:
# Import modules & datasets

import json
import numpy as np
from tensorflow.keras.datasets import mnist

In [3]:
# Load training and testing data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_valid = X_train[:10000] / 255.0
X_train = X_train[10000:] / 255.0
X_test = X_test / 255.0

X_valid = X_valid.reshape(len(X_valid), -1, 1)
X_train = X_train.reshape(len(X_train), -1, 1)
X_test = X_test.reshape(len(X_test), -1, 1)

y_valid = y_train[:10000]
y_train = y_train[10000:]

def one_hot(y):
    result = np.zeros((10, 1))
    result[y] = 1
    return result

y_train = [one_hot(y) for y in y_train]

print("Sample data loaded!")
print(f"Train data: {len(X_train)}")
print(f"Test data: {len(X_test)}")
print(f"Validation data: {len(X_valid)}")

Sample data loaded!
Train data: 50000
Test data: 10000
Validation data: 10000


In [4]:
from nnet import Network

net = Network([784, 30, 10])
print(net.sizes)

[784, 30, 10]


In [5]:
correct_guesses = net.evaluate(zip(X_valid, y_valid))
accuracy = correct_guesses / len(y_valid)
print(f"Pre-training accuracy: {correct_guesses}/{len(y_valid)} - {accuracy*100:.2f}%")

Pre-training accuracy: 985/10000 - 9.85%


In [6]:
net.sgd(
    list(zip(X_train, y_train)),
    NUM_EPOCH,
    MINI_BATCH_SIZE,
    ETA,
    list(zip(X_test, y_test))
)

10% | Epoch 3/30 | Accuracy 9159/10000
20% | Epoch 6/30 | Accuracy 9266/10000
30% | Epoch 9/30 | Accuracy 9338/10000
40% | Epoch 12/30 | Accuracy 9369/10000
50% | Epoch 15/30 | Accuracy 9396/10000
60% | Epoch 18/30 | Accuracy 9415/10000
70% | Epoch 21/30 | Accuracy 9435/10000
80% | Epoch 24/30 | Accuracy 9437/10000
90% | Epoch 27/30 | Accuracy 9426/10000
100% | Epoch 30/30 | Accuracy 9437/10000


In [7]:
print("Validating neural network...")
correct_guesses = net.evaluate(list(zip(X_valid, y_valid)))
accuracy = correct_guesses / len(y_valid)
print(f"Validation set accuracy: {correct_guesses}/{len(y_valid)} - {accuracy*100:.2f}%")

Validating neural network...
Validation set accuracy: 9450/10000 - 94.50%


In [13]:
print("Conducting analysis...")

samp = [0] * 10
acc = [0] * 10


for i in range(len(y_valid)):
    dg = y_valid[i]
    samp[dg] += 1
    acc[dg] += np.argmax(net.feedforward(X_valid[i])) == dg

for d in range(10):
    print(f"Digit {d} | {acc[d]}/{samp[d]} | {acc[d] / samp[d] * 100 :.2f}%")

print(f"Total: {sum(acc)}/{sum(samp)} | {sum(acc)/sum(samp)*100 :.2f}%")

Conducting analysis...
Digit 0 | 973/1001 | 97.20%
Digit 1 | 1094/1127 | 97.07%
Digit 2 | 915/991 | 92.33%
Digit 3 | 954/1032 | 92.44%
Digit 4 | 930/980 | 94.90%
Digit 5 | 800/863 | 92.70%
Digit 6 | 979/1014 | 96.55%
Digit 7 | 1024/1070 | 95.70%
Digit 8 | 864/944 | 91.53%
Digit 9 | 917/978 | 93.76%
Total: 9450/10000 | 94.50%


In [14]:
nnet_data = {
    "epoch": NUM_EPOCH,
    "eta": ETA,
    "mini_batch_size": MINI_BATCH_SIZE,
    "accuracy": accuracy,
    "weights": [i.tolist() for i in net.weights],
    "biases": [i.tolist() for i in net.biases]
}

with open("nnet.json", "w") as f:
    json.dump(nnet_data, f, indent=4)